# Donut masking (MaskMyPy)

Applies random-perturbation donut masking to the original point dataset using [MaskMyPy](https://github.com/maskMyPy/maskMyPy), testing four min/max displacement distance combinations and keeping the one with the lowest median displacement that still satisfies k-anonymity ≥ 5 against the population grid.

**Input:** the original point dataset and population centroids.
**Output:** the selected donut-masked shapefile in `data/donut_masking/`.


In [ ]:
# Imports
import os
from pathlib import Path
import geopandas as gpd
import pandas as pd
from maskmypy import donut, analysis


In [ ]:
#Paths
# Data directory. Override with the DATA_DIR environment variable to
# point at a different copy of the data without editing this file, e.g.
#   DATA_DIR=/path/to/data jupyter execute this_notebook.ipynb
# The data itself is not part of this repository -- see README.md.
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

ODP_PATH = DATA_DIR / "original" / "2008_eov_illness_stand.shp"
POP_PATH = DATA_DIR / "population_grid" / "GHS_Hungary_pop_centroids.shp"
POP_COL = "VALUE"

OUT_DIR = DATA_DIR / "donut_masking"
OUT_DIR.mkdir(parents=True, exist_ok=True)

#Loading ODP and population centroids
odp = gpd.read_file(ODP_PATH)
population_gdf = gpd.read_file(POP_PATH)


In [ ]:
#Parameter combinations to test (in meters)
parameter_sets = [
    (75, 600),
    (100, 700),
    (125, 800),
    (150, 900),
]

#Testing all parameter combinations and evaluating results
results = []
best = None

#Iterate through parameter combinations, apply donut mask, evaluate results, and select best configuration
for low, high in parameter_sets:
    print(f"Testing donut mask with low={low}, high={high}")

    masked_points = donut(odp, low=low, high=high)
    stats = analysis.evaluate(
        sensitive_gdf=odp,
        candidate_gdf=masked_points,
        population_gdf=population_gdf,
        population_column=POP_COL,
    )
    #Store results for all parameter combinations
    results.append({
        "low": low,
        "high": high,
        "k_min": stats["k_min"],
        "k_mean": stats["k_mean"],
        "k_med": stats["k_med"],
        "k_satisfaction_5": stats["k_satisfaction_5"],
        "displacement_med": stats["displacement_med"],
        "displacement_mean": stats["displacement_mean"],
        "central_drift": stats["central_drift"],
    })

    #only accept results with k_min >= 5, and among those select the one with the lowest median displacement
    if stats["k_min"] >= 5 and (
        best is None or stats["displacement_med"] < best["displacement_med"]
    ):
        best = {
            "low": low,
            "high": high,
            "masked_points": masked_points,
            **{key: stats[key] for key in [
                "k_min", "k_mean", "k_med", "k_satisfaction_5",
                "displacement_med", "displacement_mean", "central_drift"
            ]},
        }

#Display results for all tested parameter combinations
results_df = pd.DataFrame(results)
print("All tested parameter combinations:")
print(results_df)

#Check if any configuration met the k_min >= 5 criterion and save the best one
if best is None:
    raise ValueError(
        "No tested masking configuration reached k_min >= 5. Try larger masking distances."
    )

#Save the best masked points to a new shapefile
output_name = f"donut_points{best['low']}_high{best['high']}.shp"
best["masked_points"].to_file(OUT_DIR / output_name)

#Print summary of the best configuration
print(f"\nSelected: low={best['low']}, high={best['high']}")
print(f"k_min={best['k_min']}, displacement_med={best['displacement_med']:.3f}m")
print(f"Saved to: {OUT_DIR / output_name}")